In [56]:
import requests
import time
import urllib.parse
import pandas as pd
import os
import json
from datetime import datetime, date
from collections import defaultdict

In [57]:
# ── USER CONFIG ────────────────────────────────────────────────────────────────
STEAM_COOKIE = "76561198723087028%7C%7CeyAidHlwIjogIkpXVCIsICJhbGciOiAiRWREU0EiIH0.eyAiaXNzIjogInI6MDAwMV8yN0Y1RUMzQV82OEQxNyIsICJzdWIiOiAiNzY1NjExOTg3MjMwODcwMjgiLCAiYXVkIjogWyAid2ViOmNvbW11bml0eSIgXSwgImV4cCI6IDE3NzUyMjE0NTAsICJuYmYiOiAxNzY2NDk0MTMwLCAiaWF0IjogMTc3NTEzNDEzMCwgImp0aSI6ICIwMDAxXzI3RjVFQzNBXzY4RDZFIiwgIm9hdCI6IDE3NzUxMzQxMzAsICJydF9leHAiOiAxNzc3NzM4Njc3LCAicGVyIjogMCwgImlwX3N1YmplY3QiOiAiMTcyLjIyNS4yNDUuMTAyIiwgImlwX2NvbmZpcm1lciI6ICIxNzIuMjI1LjI0NS4xMDIiIH0._Y4Rm9GMmWyYk4-Z7RNeqGlOJ4g3JcmWNlkX2uBRyp440_HYrCwjMsqi2Y0mxar8l-gZTD1zbhtc1Y5Ze24kCw"

START_DATE = datetime.strptime("Aug 13 2013", "%b %d %Y")
END_DATE   = datetime.strptime("Mar 31 2026", "%b %d %Y")

LONG_FILE       = "CS2_long.parquet"         # intermediate store — do not delete
OUTPUT_FILE     = "CS2_Skin_Prices.csv"      # wide-format final output
CHECKPOINT_FILE = "scraper_checkpoint.json"
SLEEP_SECONDS   = 3.5                         # ~17 req/min — safe rate limit
REBUILD_EVERY   = 99999                          # rebuild Excel every N fetches
# ───────────────────────────────────────────────────────────────────────────────

HEADERS = {
    "Cookie": f"steamLoginSecure={STEAM_COOKIE}",
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
}

# Wear tiers in display order: (column label, float_lo, float_hi)
WEAR_TIERS = [
    ("(Factory New)",    0.00, 0.07),
    ("(Minimal Wear)",   0.07, 0.15),
    ("(Field-Tested)",   0.15, 0.38),
    ("(Well-Worn)",      0.38, 0.45),
    ("(Battle-Scarred)", 0.45, 1.00),
]
WEAR_ORDER = {label: i for i, (label, _, _) in enumerate(WEAR_TIERS)}

WEAPON_CATEGORIES = {
    "Pistols", "Rifles", "SMGs", "Heavy",
    "Shotguns", "Machine Guns", "Knives", "Gloves",
}

In [58]:
def valid_wears(min_float: float, max_float: float) -> list[str]:
    """Return wear labels whose float tier overlaps [min_float, max_float]."""
    return [
        label
        for label, tier_lo, tier_hi in WEAR_TIERS
        if tier_lo < max_float and tier_hi > min_float
    ]

In [59]:
def generate_item_list() -> list[str]:
    """
    Fetch the ByMykel community skin database, filter to weapon categories only,
    and expand each base skin into only its valid wear variants.
    Returns a list of Steam market hash names in grouped wear order.
    """
    print("Fetching skin database from ByMykel API...")
    resp = requests.get(
        "https://raw.githubusercontent.com/ByMykel/CSGO-API/main/public/api/en/skins.json", timeout=30
    )
    resp.raise_for_status()
    all_items = resp.json()

    seen_base: set[str] = set()
    skin_wears: dict[str, list[str]] = {}

    skipped_category = 0
    skipped_no_name  = 0

    for item in all_items:
        weapon       = item.get("weapon") or {}
        pattern      = item.get("pattern") or {}
        category_obj = item.get("category") or {}  # <--- New location in the API

        category     = category_obj.get("name", "")  # <--- Pull from category_obj
        weapon_name  = weapon.get("name", "").strip()
        pattern_name = pattern.get("name", "").strip()

        if category not in WEAPON_CATEGORIES:
            skipped_category += 1
            continue

        if weapon_name and pattern_name:
            base_name = f"{weapon_name} | {pattern_name}"
        elif weapon_name:
            base_name = weapon_name
        else:
            skipped_no_name += 1
            continue

        if category in ["Knives", "Gloves"]:
            base_name = f"★ {base_name}"

        if base_name in seen_base:
            continue

        if base_name in seen_base:
            continue
        seen_base.add(base_name)

        min_f = float(item.get("min_float") or 0.00)
        max_f = float(item.get("max_float") or 1.00)
        wears = valid_wears(min_f, max_f)
        if wears:
            skin_wears[base_name] = wears

    market_names: list[str] = [
        f"{base} {wear}"
        for base, wears in sorted(skin_wears.items())
        for wear in wears
    ]

    print(
        f"Done.  {len(seen_base)} unique base skins -> "
        f"{len(market_names)} market listings to scrape.  "
        f"(Skipped {skipped_category} non-weapon, {skipped_no_name} unnamed)"
    )
    return market_names

In [60]:
def fetch_item(item_name: str, idx: int, total: int) -> list[dict] | str:
    """
    Fetch Steam price history with overnight self-healing for timeouts and rate limits.
    """
    print(f"[{idx}/{total}] {item_name}")
    url = (
        "https://steamcommunity.com/market/pricehistory/"
        f"?appid=730&currency=1&market_hash_name={urllib.parse.quote(item_name)}"
    )

    max_retries = 5
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=20)

            # If Steam puts us in rate-limit jail, sleep for 15 minutes and try again
            if resp.status_code == 429:
                print(f"  [!!!] RATE LIMITED. Sleeping for 15 minutes... (Attempt {attempt+1}/{max_retries})")
                time.sleep(900)
                continue

            # If we get a response and it's not a 429, break out of the retry loop
            break

        except Exception as exc: # This catches your TimeoutError safely!
            if attempt < max_retries - 1:
                print(f"  [!] Timeout/Network error. Sleeping 5 minutes... (Attempt {attempt+1}/{max_retries})")
                time.sleep(300)
            else:
                print("  [!] Failed after multiple attempts. Skipping this skin.")
                return []

    # If we loop 5 times and STILL get a 429, gracefully tell the main loop to stop
    if resp.status_code == 429:
        return "RATE_LIMITED"

    try:
        data = resp.json()
    except ValueError:
        print(f"  [!] Non-JSON response (HTTP {resp.status_code})")
        return []

    if not data.get("success") or not data.get("prices"):
        print(f"  [!] No market data")
        return []

    daily: dict[date, dict] = defaultdict(lambda: {"vwap_num": 0.0, "volume": 0})

    for point in data["prices"]:
        raw_date_str = point[0][:11]       # "Mmm DD YYYY"
        price        = float(point[1])
        volume       = int(point[2])

        try:
            dt = datetime.strptime(raw_date_str, "%b %d %Y")
        except ValueError:
            continue

        if not (START_DATE <= dt <= END_DATE):
            continue

        day = dt.date()
        daily[day]["vwap_num"] += price * volume
        daily[day]["volume"]   += volume

    rows = []
    for day, agg in sorted(daily.items()):
        vol = agg["volume"]
        rows.append({
            "Skin Name":     item_name,
            "Date":          day,
            "Average Price": round(agg["vwap_num"] / vol, 4) if vol > 0 else None,
            "Volume":        vol,
        })

    return rows

In [61]:
# ── Checkpoint helpers ─────────────────────────────────────────────────────────

def load_checkpoint() -> set[str]:
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            return set(json.load(f).get("completed", []))
    return set()


def save_checkpoint(completed: set[str]) -> None:
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"completed": sorted(completed)}, f)


# ── Parquet append (long format) ───────────────────────────────────────────────

def append_to_parquet(new_rows: list[dict]) -> None:
    new_df = pd.DataFrame(new_rows)
    new_df["Date"] = pd.to_datetime(new_df["Date"])

    if os.path.exists(LONG_FILE):
        existing = pd.read_parquet(LONG_FILE)
        combined = pd.concat([existing, new_df], ignore_index=True)
        combined.drop_duplicates(subset=["Skin Name", "Date"], keep="last", inplace=True)
    else:
        combined = new_df

    combined.sort_values(["Skin Name", "Date"], inplace=True)
    combined.to_parquet(LONG_FILE, index=False)


# ── Wide-format Excel rebuild ──────────────────────────────────────────────────

def _wear_sort_key(skin_col: str) -> tuple[str, int]:
    """
    Sort key for skin column names (which may have a ' - Price' / ' - Volume'
    suffix appended). Groups alphabetically by base skin name, then orders
    wear tiers FN -> MW -> FT -> WW -> BS within each skin.
    """
    # Strip metric suffix so sorting works on the raw skin name
    col = skin_col.removesuffix(" - Price").removesuffix(" - Volume")
    for label, order in WEAR_ORDER.items():
        if col.endswith(label):
            base = col[: -len(label)].strip()
            return (base, order)
    return (col, 99)


def rebuild_wide_excel() -> None:
    """
    Read the full long-format parquet and write a fresh wide-format Excel.

    Column layout per skin/wear combination (grouped, wear order FN->BS):
        <Skin Name> - Price  |  <Skin Name> - Volume

    Rows: one per calendar day that has ANY trade data across all skins.
    Blank cells where a skin had no trades on a given day.
    """
    if not os.path.exists(LONG_FILE):
        return

    long_df = pd.read_parquet(LONG_FILE)
    long_df["Date"] = pd.to_datetime(long_df["Date"]).dt.date

    # Pivot price and volume separately
    price_wide = long_df.pivot_table(
        index="Date", columns="Skin Name", values="Average Price", aggfunc="mean"
    )
    volume_wide = long_df.pivot_table(
        index="Date", columns="Skin Name", values="Volume", aggfunc="sum"
    )

    # Rename columns to include metric suffix
    price_wide.columns  = [f"{c} - Price"  for c in price_wide.columns]
    volume_wide.columns = [f"{c} - Volume" for c in volume_wide.columns]

    # Interleave: for each skin keep Price col immediately followed by Volume col
    # Get the canonical sorted skin order from price columns
    sorted_price_cols  = sorted(price_wide.columns,  key=_wear_sort_key)
    sorted_volume_cols = sorted(volume_wide.columns, key=_wear_sort_key)

    interleaved_cols = []
    for p_col, v_col in zip(sorted_price_cols, sorted_volume_cols):
        interleaved_cols.append(p_col)
        interleaved_cols.append(v_col)

    # Join on Date index, then reorder columns
    wide = price_wide.join(volume_wide, how="outer")
    wide = wide.reindex(interleaved_cols, axis=1)
    wide.sort_index(inplace=True)

    wide.reset_index(inplace=True)
    wide.rename_axis(None, axis=1, inplace=True)

    wide.to_csv(OUTPUT_FILE, index=False)
    n_skins = len(interleaved_cols) // 2
    print(f"  -> Excel rebuilt: {len(wide)} dates x {n_skins} skins ({len(interleaved_cols)} data columns)")

In [62]:
def main():
    if STEAM_COOKIE == "YOUR_STEAM_COOKIE_HERE":
        print("ERROR: Paste your steamLoginSecure cookie value into STEAM_COOKIE above.")
        return

    all_items = generate_item_list()
    total     = len(all_items)
    completed = load_checkpoint()

    if completed:
        print(f"Resuming from checkpoint - {len(completed)}/{total} already done.")

    rate_limited    = False
    fetches_since_rebuild = 0

    for idx, item in enumerate(all_items, start=1):
        if item in completed:
            continue

        rows = fetch_item(item, idx, total)

        if rows == "RATE_LIMITED":
            rate_limited = True
            break

        if rows:
            append_to_parquet(rows)
            fetches_since_rebuild += 1

        completed.add(item)
        save_checkpoint(completed)

        # Rebuild Excel every REBUILD_EVERY successful fetches (not every single one)
        if fetches_since_rebuild >= REBUILD_EVERY:
            rebuild_wide_excel()
            fetches_since_rebuild = 0

        time.sleep(SLEEP_SECONDS)

    # Always do a final rebuild at the end
    rebuild_wide_excel()

    print()
    if rate_limited:
        print(
            f"Stopped - rate limited by Steam. "
            f"{len(completed)}/{total} done. Re-run this cell to resume."
        )
    else:
        print(f"Complete! {len(completed)} items scraped. Data in {OUTPUT_FILE}.")


main()

Fetching skin database from ByMykel API...
Done.  1933 unique base skins -> 8864 market listings to scrape.  (Skipped 7 non-weapon, 0 unnamed)
Resuming from checkpoint - 8864/8864 already done.
  -> Excel rebuilt: 4614 dates x 8738 skins (17476 data columns)

Complete! 8864 items scraped. Data in CS2_Skin_Prices.csv.
